In [1]:
#2021.08.18. WED
#Team_밥믈리에

#00. 패키지 호출
import pandas as pd 
import numpy as np 
import re 
import itertools
import csv
import konlpy
from konlpy.tag import Okt
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
import warnings 

#00-1. warning message ignore 
warnings.filterwarnings(action='ignore')

#00-2. 사전 변수 정의하기
CULTIVAR_LIST = ['고시히카리', '골드퀸', '밀키퀸', '삼광', '새누리', '새청무', '신동진', '영호진미', '오대', '일품', '진상', '참드림', '추청', '하이아미', '히토메보레'] 


In [2]:
#01. 분류값 일원화 후 저장하기.
for CULTIVAR in CULTIVAR_LIST                                                                                                         :
    morph_data = pd.read_excel(f'../data/nlp/word_frequency/{CULTIVAR}_word_frequency.xlsx')
    for i in range(len(morph_data))                                                                                                   :
        if morph_data['value'][i] in ['달다', '단맛', '달콤하다', '달달']                                                             :
            morph_data.iloc[i,0] = '달다'
        elif morph_data['value'][i] in ['담백하다']                                                                                   :
            morph_data.iloc[i,0] = '담백하다'
        elif morph_data['value'][i] in ['짜다']                                                                                       :
            morph_data.iloc[i,0] = '짜다'
        elif morph_data['value'][i] in ['구수하다', '고소하다']                                                                       :
            morph_data.iloc[i,0] = '고소하다'
        elif morph_data['value'][i] in ['눅눅하다','진밥', '질퍽', '죽밥']                                                            :
            morph_data.iloc[i,0] = '눅눅하다'
        elif morph_data['value'][i] in ['찰지다', '찰밥', '찰기', '찰짐', '촉촉하다']                                                 :
            morph_data.iloc[i,0] = '찰지다'
        elif morph_data['value'][i] in ['부드러움', '부드럽다']                                                                       :
            morph_data.iloc[i,0] = '부드럽다'
        elif morph_data['value'][i] in ['쫄깃', '쫄깃쫄깃', '쫀득해', '쫀득하', '쫀득', '쫀득쫀득해', '쫀득쫀득', '쫀득거리', '쫀쫀'] :
            morph_data.iloc[i,0] = '쫀득하다'
        elif morph_data['value'][i] in ['고슬고슬', '고실고실', '고슬고슬하다', '고슬', '꼬들꼬들', '고슬밥', '꼬두밥', '꼬들밥']     :
            morph_data.iloc[i,0] = '고슬하다' 
        elif morph_data['value'][i] in ['푸석거리다', '퍼석퍼석', '푸석하다', '푸석푸석하다', '푸석푸석', '푸석', '떨어지다', '흩어지다' \
                                        ,'날아가다', '부스러지다', '날아다니다', '딱딱하다' \
                                        , '퍼석거리다', '퍽퍽해', '퍽퍽'] :
            morph_data.iloc[i,0] = '푸석하다'
        elif morph_data['value'][i] in ['통통하다', '탱탱하다', '탱글탱글하다', '탱글탱글', '통통', '두껍다', '굵직하다']             :
            morph_data.iloc[i,0] = '통통하다'
        elif morph_data['value'][i] in ['매끈', '매끈하다', '반지르르']                                                               :
            morph_data.iloc[i,0] = '매끈하다'
        elif morph_data['value'][i] in ['균일하다']                                                                                   :
            morph_data.iloc[i,0] = '균일하다'
        elif morph_data['value'][i] in ['윤기', '지다', '누룽지', '초밥', '볶음밥', '누룽', '유아식', '햅살', '시큼하다', '싱겁다']   :
            morph_data.iloc[i,0] = 'delete_value'
            
    morph_data_grouping = morph_data.groupby('value').sum()
    morph_data_grouping.reset_index(inplace=True)
    morph_data_grouping = morph_data_grouping[morph_data_grouping['value'] != 'delete_value']
    morph_data_grouping.columns = ['value', f'count_{CULTIVAR}']
    morph_data_grouping.to_excel(f'../data/nlp/indicator_grouping/{CULTIVAR}_grouping.xlsx', index=False)
    print(f'{CULTIVAR} DONE !')

고시히카리 DONE !
골드퀸 DONE !
밀키퀸 DONE !
삼광 DONE !
새누리 DONE !
새청무 DONE !
신동진 DONE !
영호진미 DONE !
오대 DONE !
일품 DONE !
진상 DONE !
참드림 DONE !
추청 DONE !
하이아미 DONE !
히토메보레 DONE !


In [3]:
#02. 각 데이터셋 재 할당 후 병합, 저장하기.
#(1) 저장한 데이터셋 불러와서 동적으로 할당하기
for CULTIVAR in CULTIVAR_LIST :
    globals()[f'{CULTIVAR}_data'] = pd.read_excel(f'../data/nlp/indicator_grouping/{CULTIVAR}_grouping.xlsx')

#(2) 각 데이터셋 병합하기. 
total_data = 고시히카리_data
for CULTIVAR in CULTIVAR_LIST[1:] :
    total_data = pd.merge(total_data, globals()[f'{CULTIVAR}_data'], on='value', how='outer')

#(3) 확인하기. 
total_data

,value,count_고시히카리,count_골드퀸,count_밀키퀸,count_삼광,count_새누리,count_새청무,count_신동진,count_영호진미,count_오대,count_일품,count_진상,count_참드림,count_추청,count_하이아미,count_히토메보레
0,고소하다,185.0,235.0,NaN,146.0,NaN,6.0,296,73.0,98.0,27.0,12.0,24.0,194.0,NaN,13.0
1,고슬하다,55.0,NaN,NaN,26.0,NaN,NaN,52,18.0,22.0,NaN,NaN,NaN,38.0,NaN,NaN
2,눅눅하다,18.0,NaN,NaN,NaN,NaN,NaN,26,9.0,13.0,NaN,NaN,NaN,31.0,NaN,NaN
3,달다,117.0,NaN,NaN,65.0,NaN,NaN,116,16.0,47.0,NaN,13.0,NaN,103.0,NaN,9.0
4,매끈하다,20.0,NaN,NaN,25.0,NaN,NaN,296,7.0,28.0,NaN,NaN,NaN,40.0,NaN,NaN
5,부드럽다,37.0,NaN,NaN,20.0,NaN,NaN,47,21.0,15.0,NaN,NaN,NaN,62.0,NaN,NaN
6,짜다,55.0,8.0,NaN,32.0,NaN,NaN,109,34.0,31.0,8.0,7.0,NaN,91.0,NaN,NaN
7,쫀득하다,147.0,6.0,NaN,65.0,NaN,NaN,115,51.0,44.0,NaN,15.0,NaN,57.0,NaN,NaN
8,찰지다,458.0,26.0,NaN,266.0,NaN,6.0,612,238.0,210.0,41.0,54.0,25.0,384.0,NaN,6.0
9,통통하다,75.0,NaN,NaN,22.0,NaN,NaN,139,5.0,31.0,NaN,NaN,NaN,52.0,NaN,NaN


In [4]:
#(4) 병합 데이터셋 저장하기.
total_data.to_excel('../data/nlp/total_merge_data.xlsx', index=False)

In [5]:
#03. 병합 데이터셋 스케일링 처리하기. 
#(1) Min-Max Scaler 객체 할당하기.
scaler = MinMaxScaler()

#(2) 스케일링 처리하기. 
total_data_scaling = scaler.fit_transform(total_data.iloc[:,1:])

#(3) 데이터프레임으로 변환하기. 
total_data_scaling = pd.DataFrame(total_data_scaling, columns= total_data.columns[1:])
total_data_scaling['value'] = total_data['value']
total_data_scaling = total_data_scaling[total_data.columns]
total_data_scaling

,value,count_고시히카리,count_골드퀸,count_밀키퀸,count_삼광,count_새누리,count_새청무,count_신동진,count_영호진미,count_오대,count_일품,count_진상,count_참드림,count_추청,count_하이아미,count_히토메보레
0,고소하다,0.379545,1.000000,NaN,0.534884,NaN,0.0,0.478548,0.291845,0.431472,0.588235,0.106383,0.0,0.497354,NaN,1.000000
1,고슬하다,0.084091,NaN,NaN,0.069767,NaN,NaN,0.075908,0.055794,0.045685,NaN,NaN,NaN,0.084656,NaN,NaN
2,눅눅하다,0.000000,NaN,NaN,NaN,NaN,NaN,0.033003,0.017167,0.000000,NaN,NaN,NaN,0.066138,NaN,NaN
3,달다,0.225000,NaN,NaN,0.220930,NaN,NaN,0.181518,0.047210,0.172589,NaN,0.127660,NaN,0.256614,NaN,0.428571
4,매끈하다,0.004545,NaN,NaN,0.065891,NaN,NaN,0.478548,0.008584,0.076142,NaN,NaN,NaN,0.089947,NaN,NaN
5,부드럽다,0.043182,NaN,NaN,0.046512,NaN,NaN,0.067657,0.068670,0.010152,NaN,NaN,NaN,0.148148,NaN,NaN
6,짜다,0.084091,0.008734,NaN,0.093023,NaN,NaN,0.169967,0.124464,0.091371,0.029412,0.000000,NaN,0.224868,NaN,NaN
7,쫀득하다,0.293182,0.000000,NaN,0.220930,NaN,NaN,0.179868,0.197425,0.157360,NaN,0.170213,NaN,0.134921,NaN,NaN
8,찰지다,1.000000,0.087336,NaN,1.000000,NaN,0.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,NaN,0.000000
9,통통하다,0.129545,NaN,NaN,0.054264,NaN,NaN,0.219472,0.000000,0.091371,NaN,NaN,NaN,0.121693,NaN,NaN


In [6]:
#(4) 병합 데이터셋 저장하기.
total_data_scaling.to_excel('../data/nlp/total_merge_data_scaling.xlsx', index=False)

In [7]:
#04. 병합 데이터셋 행, 열 변환하기(전치행렬)
#(1) 전치 처리하기.
final_dataset = total_data_scaling.T

#(2) 인덱스 컬럼으로 반환 후 처리하기
final_dataset.reset_index(inplace=True)

#(3) 컬럼 명 변경하기. 
final_dataset.columns = final_dataset.iloc[0,:]

#(4) 불필요 행 삭제하기.
final_dataset = final_dataset.iloc[1:,]

#(5) 처리 확인하기. 
final_dataset

,value,고소하다,고슬하다,눅눅하다,달다,매끈하다,부드럽다,짜다,쫀득하다,찰지다,통통하다,푸석하다,균일하다,담백하다
1,count_고시히카리,0.379545,0.084091,0.0,0.225,0.004545,0.043182,0.084091,0.293182,1.0,0.129545,0.104545,NaN,NaN
2,count_골드퀸,1.0,NaN,NaN,NaN,NaN,NaN,0.008734,0.0,0.087336,NaN,NaN,NaN,NaN
3,count_밀키퀸,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,count_삼광,0.534884,0.069767,NaN,0.22093,0.065891,0.046512,0.093023,0.22093,1.0,0.054264,0.162791,0.0,NaN
5,count_새누리,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,count_새청무,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
7,count_신동진,0.478548,0.075908,0.033003,0.181518,0.478548,0.067657,0.169967,0.179868,1.0,0.219472,0.257426,0.008251,0.0
8,count_영호진미,0.291845,0.055794,0.017167,0.04721,0.008584,0.06867,0.124464,0.197425,1.0,0.0,0.339056,NaN,NaN
9,count_오대,0.431472,0.045685,0.0,0.172589,0.076142,0.010152,0.091371,0.15736,1.0,0.091371,0.263959,NaN,NaN
10,count_일품,0.588235,NaN,NaN,NaN,NaN,NaN,0.029412,NaN,1.0,NaN,0.0,NaN,NaN


In [8]:
#(6) 최종 데이터셋 저장하기. 
final_dataset.to_excel('../data/NLP/cultivar_MF.xlsx', index=False) 